# CF and UGRID Standards
## xarray-cf and xugrid packages

**Duration: ~30 minutes**

This notebook covers two important metadata conventions for scientific datasets and the Python packages that leverage them:

| Convention | Scope | Python Package |
|---|---|---|
| [CF Conventions](https://cfconventions.org/) | Variable semantics (names, units, axes, coordinates) | [`cf_xarray`](https://cf-xarray.readthedocs.io/) |
| [UGRID Conventions](https://ugrid-conventions.github.io/ugrid-conventions/) | Unstructured mesh topology (nodes, edges, faces) | [`xugrid`](https://deltares.github.io/xugrid/) |

### Why do conventions matter?

Without conventions, accessing the longitude coordinate of a dataset might require knowing its exact variable name — `lon`, `longitude`, `LON`, `x`, `nav_lon` — which varies by model or instrument. CF Conventions standardize this through metadata attributes (`standard_name`, `axis`, `units`), enabling **interoperable, self-describing datasets**.

UGRID extends CF for **unstructured grids** (triangular meshes, irregular networks) common in ocean and coastal models like FVCOM, SCHISM, and SHYFEM.

---
## Part 1: CF Conventions and cf_xarray (≈15 min)

### 1.1 What are CF Conventions?

The [Climate and Forecast (CF) Metadata Conventions](https://cfconventions.org/) are a set of guidelines for describing scientific data stored in NetCDF (and increasingly Zarr) files. Originally developed for atmospheric and ocean models, they are now widely used across Earth sciences.

Key attributes defined by CF:

| Attribute | Purpose | Example |
|---|---|---|
| `standard_name` | Canonical variable identity | `"sea_water_temperature"` |
| `long_name` | Human-readable description | `"Sea Water Temperature at 10m"` |
| `units` | Physical units (UDUNITS compatible) | `"degC"`, `"m s-1"`, `"days since 1970-01-01"` |
| `axis` | Coordinate type | `"X"`, `"Y"`, `"Z"`, `"T"` |
| `positive` | Vertical direction | `"up"` or `"down"` |
| `coordinates` | Associated coordinate variables | `"lon lat depth time"` |
| `cell_methods` | Statistical operations | `"time: mean"` |
| `cell_measures` | Area/volume weights | `"area: areacello"` |

The `Conventions` global attribute declares compliance, e.g. `"CF-1.8"`.

In [ ]:
import xarray as xr
import cf_xarray
import numpy as np
import hvplot.xarray

# Load a built-in xarray tutorial dataset
ds = xr.tutorial.open_dataset('air_temperature')
ds

### 1.2 Inspecting CF attributes

Let's look at the metadata that CF conventions provide on the `air` variable and its coordinates.

In [ ]:
# Inspect CF attributes on the main variable
print("=== 'air' variable attributes ===")
for k, v in ds['air'].attrs.items():
    print(f"  {k}: {v}")

print("\n=== 'time' coordinate attributes ===")
for k, v in ds['time'].attrs.items():
    print(f"  {k}: {v}")

### 1.3 The `cf` accessor from cf_xarray

Importing `cf_xarray` adds a `.cf` accessor to all xarray objects.  
This lets you refer to coordinates **by role** rather than by name.

In [ ]:
# What CF roles does cf_xarray detect in this dataset?
ds.cf

In [ ]:
# Access coordinates by CF axis name instead of variable name
lon = ds.cf['longitude']
lat = ds.cf['latitude']
t   = ds.cf['time']

print(f"longitude variable name: '{lon.name}'")
print(f"latitude  variable name: '{lat.name}'")
print(f"time      variable name: '{t.name}'")

This works even when the variable is named `lon_rho`, `nav_lon`, or `x` — as long as the CF attribute `axis: X` or `standard_name: longitude` is present.  
This is the key interoperability benefit.

### 1.4 Selection and subsetting with `cf.sel()`

Use axis keys (`X`, `Y`, `Z`, `T`) or standard names in place of coordinate variable names.

In [ ]:
# Select a time slice using CF axis 'T' rather than the variable name 'time'
ds_jan = ds.cf.sel(T='2013-01', method='nearest')
ds_jan

In [ ]:
# Time mean using CF axis name — works regardless of the time variable name
ds_mean = ds.mean(ds.cf.axes['T'])
ds_mean['air'].hvplot.quadmesh(
    x='lon', y='lat', rasterize=True, geo=True, tiles='OSM',
    cmap='RdBu_r', title='Mean Air Temperature (K)'
)

### 1.5 Real-world example: ROMS/COAWST ocean model

The COAWST dataset uses non-standard coordinate names (`lon_rho`, `lat_rho`, `ocean_time`, `s_rho`), but CF attributes allow `cf_xarray` to find them automatically.

In [ ]:
import intake
import zarr

intake_catalog_url = 'https://usgs-coawst.s3.amazonaws.com/useast-archive/coawst_intake.yml'
cat = intake.open_catalog(intake_catalog_url)

if zarr.__version__[0] == '2':
    dataset = 'COAWST-USEAST-zarr2'
else:
    dataset = 'COAWST-USEAST-zarr3'

ds_roms = cat[dataset].to_dask()
ds_roms

In [ ]:
# With CF, no need to know the exact coordinate names
da = ds_roms['Hwave']

x = da.cf['longitude']
y = da.cf['latitude']
t = da.cf['time']

print(f"longitude → '{x.name}',  latitude → '{y.name}',  time → '{t.name}'")

# Select using CF axis 'T' — works regardless of the time variable name
da_slice = da.cf.sel(T='2012-10-29 12:00', method='nearest').load()
da_slice.hvplot.quadmesh(x=x.name, y=y.name, rasterize=True, geo=True,
                         tiles='OSM', cmap='plasma', title='Significant Wave Height (m)')

### 1.6 cf_xarray: additional capabilities

Beyond coordinate lookup, `cf_xarray` also supports:

- **`ds.cf.describe()`** — human-readable summary of all CF roles detected
- **`ds.cf.bounds`** — access cell boundary variables
- **`ds.cf.cell_measures`** — find area/volume weight arrays (e.g. `areacello`)
- **`ds.cf.add_bounds()`** — auto-generate 1D cell bounds from center coordinates
- **`ds.cf.guess_coord_axis()`** — infer axes from coordinate names/units when attributes are missing
- **`cf_xarray.units`** — integrate with `pint` for unit-aware calculations

In [ ]:
# Summarize all CF roles detected in the air_temperature dataset
ds.cf.describe()

---
## Part 2: UGRID Conventions and xugrid (≈15 min)

### 2.1 What are UGRID Conventions?

[UGRID](https://ugrid-conventions.github.io/ugrid-conventions/) extends CF Conventions to describe **unstructured meshes** — grids where cells are not arranged in a regular lat/lon matrix.

**Why unstructured grids?**

| Structured Grid | Unstructured Grid |
|---|---|
| Regular lat/lon or curvilinear | Triangles, quadrilaterals, polygons |
| Easy to index `[i, j]` | Requires topology connectivity |
| Wastes cells on land | Concentrates resolution where needed |
| ROMS, MOM, NEMO | FVCOM, SCHISM, SHYFEM, Delft3D FM |

**UGRID topology elements:**

```
node  — the points (vertices) of the mesh
edge  — line segments connecting two nodes
face  — polygonal cells (triangles, quads, ...)
```

**Key connectivity arrays** (stored as CF variables):

| Variable | Meaning |
|---|---|
| `face_node_connectivity` | which nodes form each face |
| `edge_node_connectivity` | which nodes form each edge |
| `face_edge_connectivity` | which edges bound each face |

A **mesh topology variable** (a dummy scalar with `cf_role: mesh_topology`) declares the mesh and references these connectivity arrays.

### 2.2 The xugrid package

[xugrid](https://deltares.github.io/xugrid/) (by Deltares) extends xarray to work natively with UGRID datasets.  
It wraps `xr.Dataset` with grid-aware indexing, selection, and regridding.

Key objects:
- `xugrid.Ugrid1d` — 1D network (edges only)
- `xugrid.Ugrid2d` — 2D mesh (faces, edges, nodes)
- `xugrid.UgridDataset` / `xugrid.UgridDataArray` — xarray wrappers with `.ugrid` accessor

In [ ]:
import xugrid as xu
import matplotlib.pyplot as plt

# xugrid ships with built-in example datasets
print(dir(xu.data))

In [ ]:
# Load the built-in disk example (hydraulic model on unstructured mesh)
uds = xu.data.disk()
uds

### 2.3 Inspecting the UGRID topology

In [ ]:
# Access the underlying Ugrid2d grid object
grid = uds.ugrid.grid
print(f"Grid type: {type(grid).__name__}")
print(f"Number of nodes: {grid.n_node}")
print(f"Number of edges: {grid.n_edge}")
print(f"Number of faces: {grid.n_face}")

In [ ]:
# Plot the mesh topology
fig, ax = plt.subplots(figsize=(5, 5))
grid.plot(ax=ax)
ax.set_title('UGRID mesh topology')
ax.set_aspect('equal')
plt.show()

### 2.4 Data on the mesh

Data variables can be located at **nodes**, **edges**, or **face centroids** — each has different dimensions.

In [ ]:
# Show data variables and their dimensions
for name, da in uds.items():
    print(f"{name:20s}  dims={da.dims}")

In [ ]:
# Plot face-centroid data on the native mesh
uds['face_z'].ugrid.plot(cmap='viridis')
plt.title('face_z on native unstructured mesh')
plt.show()

In [ ]:
# Node-centered data
uds['node_z'].ugrid.plot(cmap='plasma')
plt.title('node_z on native unstructured mesh')
plt.show()

### 2.5 Selection and clipping

xugrid supports spatial selection that respects mesh topology.

In [ ]:
# Clip to a bounding box — topology is automatically subsetted
uds_clip = uds.ugrid.clip_box(xmin=-0.5, ymin=-0.5, xmax=0.5, ymax=0.5)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
uds['face_z'].ugrid.plot(ax=axes[0], cmap='viridis')
axes[0].set_title('Full domain')
uds_clip['face_z'].ugrid.plot(ax=axes[1], cmap='viridis')
axes[1].set_title('Clipped to bbox')
plt.tight_layout()
plt.show()

### 2.6 Regridding to a structured grid

A common workflow: **unstructured model output → regular grid** for post-processing, comparison, or export.

In [ ]:
# Create a target structured grid from the bounding box of the source
target_grid = xu.Ugrid2d.from_structured(
    np.linspace(-1, 1, 50), np.linspace(-1, 1, 50)
)
regridder = xu.CentroidLocatorRegridder(source=uds, target=target_grid)
da_regridded = regridder.regrid(uds['face_z'])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
uds['face_z'].ugrid.plot(ax=axes[0], cmap='viridis')
axes[0].set_title('Original unstructured')
da_regridded.ugrid.plot(ax=axes[1], cmap='viridis')
axes[1].set_title('Regridded to structured')
plt.tight_layout()
plt.show()

### 2.7 Reading UGRID-compliant NetCDF files

Any NetCDF file that follows the UGRID convention can be opened directly with `xu.open_dataset()`.  
xugrid detects the `mesh_topology` variable and builds the grid automatically.

In [ ]:
# Open a UGRID NetCDF file — xugrid auto-detects topology
#   uds = xu.open_dataset('my_model_output.nc')

# Demonstrate round-trip: export back to plain xr.Dataset with full UGRID CF attributes
ds_ugrid = uds.ugrid.to_dataset()
print("mesh_topology variable cf_role:", ds_ugrid['mesh'].attrs.get('cf_role'))
print("face_node_connectivity:", ds_ugrid['mesh'].attrs.get('face_node_connectivity'))

### 2.8 xarray operations preserve the grid

`UgridDataArray` / `UgridDataset` support most standard xarray operations.  
When an operation preserves the mesh structure, the UGRID grid is carried along automatically.

In [ ]:
# Load time-dependent example
uds_adaguc = xu.data.adaguc_dataset()
uds_adaguc

In [ ]:
# Select a time step — UGRID topology is preserved
da_t0 = uds_adaguc['salinity'].isel(time=0)
da_t0.ugrid.plot(cmap='Blues', figsize=(8, 4))
plt.title('Salinity at t=0 (native UGRID mesh)')
plt.show()

In [ ]:
# Time mean — standard xarray operation, result is still a UgridDataArray
da_mean = uds_adaguc['salinity'].mean('time')
print(type(da_mean))
da_mean.ugrid.plot(cmap='Blues', figsize=(8, 4))
plt.title('Time-mean salinity (native UGRID mesh)')
plt.show()

---
## Summary

### CF Conventions + cf_xarray

| Task | Without CF | With cf_xarray |
|---|---|---|
| Get longitude | `ds['lon_rho']` | `ds.cf['longitude']` |
| Select by time | `ds.sel(ocean_time='2012-01')` | `ds.cf.sel(T='2012-01')` |
| Find time axis | know the name | `ds.cf.axes['T']` |
| Works across models | No | Yes |

### UGRID Conventions + xugrid

| Task | Without xugrid | With xugrid |
|---|---|---|
| Open UGRID file | `xr.open_dataset()` (raw arrays) | `xu.open_dataset()` (topology-aware) |
| Plot on native mesh | manual triangulation | `.ugrid.plot()` |
| Spatial clip | manual index math | `.ugrid.clip_box()` |
| Regrid to structured | scipy + manual | `xu.CentroidLocatorRegridder` |
| Time operations | lose grid context | grid preserved automatically |

### Resources

- [CF Conventions](https://cfconventions.org/) — the standard
- [cf_xarray docs](https://cf-xarray.readthedocs.io/) — full API reference
- [UGRID Conventions](https://ugrid-conventions.github.io/ugrid-conventions/) — the standard
- [xugrid docs](https://deltares.github.io/xugrid/) — full API reference
- [xugrid examples gallery](https://deltares.github.io/xugrid/examples/)